
# HERA Topography Raster Toolkit — **Extended, Copy‑Paste Tutorial**

This notebook is an **ultra‑detailed** guide for non‑experts. It explains each step,
why it matters, common pitfalls, and gives **copy‑paste** code blocks you can run as‑is.

**Scope covered (and more than the original):**
- Safe initialization of the **Topography Raster Toolkit**
- Elevation for a single point and a list of points
- Optional xarray‑based elevation lookups
- STL generation for a region of interest (ROI)
- Basic statistics (if available in your build)
- Clear troubleshooting for mappings, kernels, and data sources



## Table of Contents
1. [Prerequisites (What you need)](#prereq)
2. [How toolkit discovery works (Why the error happens)](#discovery)
3. [Environment & kernel checks](#env)
4. [Safe toolkit loader (drop‑in)](#safe)
5. [Configure your DEM path](#dem)
6. [Define sample coordinates](#coords)
7. [Single‑point elevation](#single)
8. [Multi‑point elevation](#multi)
9. [Optional: xarray lookup](#xarray)
10. [STL generation (ROI)](#stl)
11. [Optional: Basic statistics](#stats)
12. [Troubleshooting & FAQ](#trouble)
13. [Appendix: Practical tips & Glossary](#appendix)



<a id="prereq"></a>
## 1) Prerequisites (What you need)

- **Python environment**: Use the `heraenv` interpreter (e.g., `/home/you/hera/heraenv/bin/python`).
- **HERA installed/importable**: Either `pip install -e ~/hera` or add `~/hera` to your `PYTHONPATH`.
- **hera-data mapping is in sync** with your code. For the Topography toolkit, the mapping key
  `GIS_RASTER_TOPOGRAPHY` should point to:
  ```text
  hera.measurements.GIS.raster.topography.TopographyToolkit
  ```
- **A DEM (elevation raster)** file available (e.g., HGT or GeoTIFF). We'll configure its path.
- (Optional) **xarray** if you plan to use the xarray demo.



<a id="discovery"></a>
## 2) How toolkit discovery works (and why the `'NoneType' is not callable` error happens)

- HERA uses a *mapping* (often from **hera‑data**) from a **key** to a **class path**.
- When you call:
  ```python
  from hera import toolkitHome
  tk = toolkitHome.getToolkit(toolkitName=toolkitHome.GIS_RASTER_TOPOGRAPHY)
  ```
  under the hood, it resolves the class path (e.g., `"hera.measurements.GIS.raster.topography.TopographyToolkit"`)
  and dynamically imports it.
- If the mapping is **out of date** or the class path is wrong, `pydoc.locate(...)` returns **`None`**, and
  the code tries to call `None(...)`, producing:
  ```text
  TypeError: 'NoneType' is not callable
  ```
- The fix is simply: keep **hera-data** in sync, and add a **guard** that checks the class path before instantiating.
  This notebook provides that guard so you get a helpful message instead of a cryptic crash.



<a id="env"></a>
## 3) Environment & kernel checks

Run the cell below first. It verifies you're on the intended interpreter, and that the toolkit class
can be resolved. If it fails, follow the inline instructions.


In [1]:

import sys, os, pydoc
print("Python interpreter:", sys.executable)
print("Working directory:", os.getcwd())

cls = pydoc.locate("hera.measurements.GIS.raster.topography.TopographyToolkit")
print("TopographyToolkit resolvable? ->", cls is not None)
if cls is None:
    raise ImportError(
        "Could not resolve 'TopographyToolkit'. "
        "Make sure HERA is importable (e.g., `pip install -e ~/hera`) "
        "and the hera-data mapping for GIS_RASTER_TOPOGRAPHY points to "
        "'hera.measurements.GIS.raster.topography.TopographyToolkit'."
    )


Python interpreter: /home/ilay/hera/heraenv/bin/python
Working directory: /home/ilay/hera
TopographyToolkit resolvable? -> True



<a id="safe"></a>
## 4) Safe toolkit loader (drop‑in)

This wrapper ensures the class path exists **before** creating the toolkit.  
Use it as a **drop‑in replacement** for direct `getToolkit(...)` calls.


In [2]:

from hera import toolkitHome
import pydoc

def safe_get_toolkit(*, projectName=None, filesDirectory=None, **kwargs):
    """Return a Topography toolkit instance; fail with a clear message if mapping is wrong."""
    key = getattr(toolkitHome, "GIS_RASTER_TOPOGRAPHY", None)
    mapping = getattr(toolkitHome, "toolkitsMapping", None) or getattr(toolkitHome, "_toolkitsMapping", None)
    if mapping and key in mapping:
        entry = mapping[key]
        if isinstance(entry, dict):
            clsName = entry.get("class") or entry.get("cls") or entry.get("className")
        else:
            clsName = str(entry)
    else:
        clsName = "hera.measurements.GIS.raster.topography.TopographyToolkit"
    if pydoc.locate(clsName) is None:
        raise ImportError(
            f"Could not locate toolkit class '{clsName}'. "
            "Update hera-data mapping for GIS_RASTER_TOPOGRAPHY or ensure HERA is importable."
        )
    return toolkitHome.getToolkit(toolkitName=toolkitHome.GIS_RASTER_TOPOGRAPHY,
                                  projectName=projectName, filesDirectory=filesDirectory, **kwargs)

# Instantiate once and reuse:
tk = safe_get_toolkit(projectName="TopographyDemo", filesDirectory=None)
print("Toolkit instance:", type(tk))


Toolkit instance: <class 'hera.measurements.GIS.raster.topography.TopographyToolkit'>



<a id="dem"></a>
## 5) Configure your DEM path

Pick **one** path below and edit it to a real file on your machine.  
If your toolkit auto‑discovers data sources, you can keep this as reference only.


In [3]:

# Examples (choose ONE and edit the path):
# dem_path = "/data/elevation/N33E035.SRTMGL1.hgt"
# dem_path = "/data/elevation/dem_33_35.tif"
dem_path = "/path/to/your/elevation_file.hgt"  # <-- EDIT ME

import os
if not os.path.exists(dem_path):
    print("WARNING: DEM file not found yet. Set 'dem_path' to a real file before running the examples.")



<a id="coords"></a>
## 6) Define sample coordinates

We’ll use one point and a small list for demonstrations. Replace with your own as needed.


In [4]:

lat, lon = 32.0809, 34.7806  # Tel Aviv (approx)
points = [(32.08, 34.77), (32.09, 34.79), (32.10, 34.80)]
print("Single point:", (lat, lon))
print("Point list:  ", points)


Single point: (32.0809, 34.7806)
Point list:   [(32.08, 34.77), (32.09, 34.79), (32.1, 34.8)]



<a id="single"></a>
## 7) Single‑point elevation

**What it does:** Returns the elevation value at a single `(lat, lon)` location.  
**Common pitfalls:** DEM not found, point outside the DEM tile, NoData values.


In [5]:

try:
    value = tk.getElevation(lat=lat, lon=lon)  # adjust param names if your API differs
    print(f"Elevation at ({lat:.5f}, {lon:.5f}) = {value}")
except Exception as e:
    print("getElevation failed:", e)
    print("Tip: if your toolkit expects the DEM path explicitly, check a 'dataSourceName'/'demPath' parameter or a setter.")


getElevation failed: getElevation() got an unexpected keyword argument 'lat'
Tip: if your toolkit expects the DEM path explicitly, check a 'dataSourceName'/'demPath' parameter or a setter.



<a id="multi"></a>
## 8) Multi‑point elevation

**What it does:** Returns elevations for several `(lat, lon)` pairs.  
If your build has `getPointListElevation`, prefer it; otherwise we loop over `getElevation`.


In [6]:

try:
    if hasattr(tk, "getPointListElevation"):
        values = tk.getPointListElevation(points)  # list of (lat, lon)
        print("Point list elevations:", values)
    else:
        values = [tk.getElevation(lat=p[0], lon=p[1]) for p in points]
        print("Point list elevations (looped):", values)
except Exception as e:
    print("Multi-point elevation failed:", e)


Multi-point elevation failed: 'list' object has no attribute 'shape'



<a id="xarray"></a>
## 9) Optional: xarray lookup

Some builds provide `getElevationOfXarray` for integration with scientific stacks.  
If not available, skip this section.


In [7]:

if hasattr(tk, "getElevationOfXarray"):
    try:
        import xarray as xr
        lats = [p[0] for p in points]
        lons = [p[1] for p in points]
        ds = xr.Dataset({"lat": ("n", lats), "lon": ("n", lons)})
        elev = tk.getElevationOfXarray(ds)
        print("Elevation of Xarray result:", elev)
    except Exception as e:
        print("getElevationOfXarray failed:", e)
else:
    print("Toolkit has no 'getElevationOfXarray' in this version — skipping.")


getElevationOfXarray failed: 'defaultSRTM'



<a id="stl"></a>
## 10) STL generation (ROI)

**What it does:** Creates a 3D model (STL) from an ROI around your point.  
**Advice:** Start with a small ROI to keep files light. Ensure output directory is writable.


In [8]:

import os
out_dir = os.path.abspath("./topography_outputs")
os.makedirs(out_dir, exist_ok=True)

bbox = {
    "min_lat": lat - 0.02,
    "max_lat": lat + 0.02,
    "min_lon": lon - 0.02,
    "max_lon": lon + 0.02,
}
stl_path = os.path.join(out_dir, "roi.stl")

if hasattr(tk, "createElevationSTL"):
    try:
        tk.createElevationSTL(bbox=bbox, outputPath=stl_path)  # adjust arg names to your API
        print("STL created at:", stl_path)
    except Exception as e:
        print("createElevationSTL failed:", e)

elif hasattr(tk, "getElevationSTL"):
    try:
        data = tk.getElevationSTL(bbox=bbox)  # may return bytes; write to file
        if isinstance(data, (bytes, bytearray)):
            with open(stl_path, "wb") as f:
                f.write(data)
            print("STL created at:", stl_path)
        else:
            print("getElevationSTL returned:", type(data))
    except Exception as e:
        print("getElevationSTL failed:", e)
else:
    print("Toolkit has no STL method in this version — skipping.")


createElevationSTL failed: createElevationSTL() got an unexpected keyword argument 'bbox'



<a id="stats"></a>
## 11) Optional: Basic statistics

If your build exposes `calculateStastics`/`calculateStatistics`, this computes summary stats for given points.


In [9]:

cand = ["calculateStastics", "calculateStatistics"]
name = next((n for n in cand if hasattr(tk, n)), None)
if name:
    try:
        fn = getattr(tk, name)
        res = fn(points=points)  # adjust arg names to your API
        print("Statistics:", res)
    except Exception as e:
        print(f"{name} failed:", e)
else:
    print("No statistics method on this toolkit version — skipping.")


No statistics method on this toolkit version — skipping.



<a id="trouble"></a>
## 12) Troubleshooting & FAQ

**Q: I get `TypeError: 'NoneType' is not callable`.**  
A: Your mapping for `GIS_RASTER_TOPOGRAPHY` is wrong/out of sync. Use the safe loader in this notebook and update
   hera-data so it points to `hera.measurements.GIS.raster.topography.TopographyToolkit`.

**Q: Kernel errors in PyCharm (e.g., “Kernel does not exist”).**  
A: Start a **new** kernel using the `heraenv` interpreter. If needed, clear stale kernelspecs:
   - In PyCharm, configure a *Managed Server* using the project interpreter (`heraenv`).
   - Or connect to an *Existing Server* with the full URL (include `?token=...`, no trailing `/lab`).

**Q: DEM file not found.**  
A: Set `dem_path` to a real file, check permissions, ensure the point(s) are inside the DEM extent.

**Q: STL creation is slow/huge.**  
A: Shrink the ROI (`bbox`), or pass resolution/scale parameters if your API supports them.

**Q: Some points return None/NaN.**  
A: Many DEMs (e.g., HGT) have NoData pixels. Filter or fill those values before doing statistics.



<a id="appendix"></a>
## 13) Appendix — Practical tips & Glossary

- **DEM**: Digital Elevation Model. A raster grid where each cell is an elevation value.
- **HGT / GeoTIFF**: Common file formats for DEMs.
- **ROI (Region of Interest)**: The spatial subset you process (we used a simple bbox).
- **STL**: A triangle mesh format used by 3D printers and many 3D viewers.
- **Mapping**: hera‑data’s way to map a toolkit key to a concrete Python class path.
- **pydoc.locate**: Routine used to resolve a `"package.module.Class"` string to the actual class object.

**Recommended workflow:** start small → confirm single‑point elevation → try a few points → only then generate STL over a larger ROI.
